In [16]:
import json

In [16]:

def load_data(filename):
    with open(filename) as f:
        return json.load(f)

In [17]:


def pags_you_might_like_pandas(target_user_id, data):
    # 1. Flatten the JSON data into a clean DataFrame
    records = []
    for user in data["users"]:
        uid = user["id"]
        for page in user["liked_pages"]:
            records.append({"user_id": uid, "page_id": page})

    df = pd.DataFrame(records)

    # If target user doesn't exist or data is empty, exit early
    if df.empty or target_user_id not in df["user_id"].values:
        return []

    # 2. Get target user pages
    target_pages = set(df[df["user_id"] == target_user_id]["page_id"])

    # 3. Create a user-page interaction set dictionary
    user_pages_dict = df.groupby("user_id")["page_id"].apply(set).to_dict()

    page_suggestions = {}

    # 4. Compute Jaccard Similarity weights
    for other_user_id, other_pages in user_pages_dict.items():
        if other_user_id == target_user_id:
            continue

        intersection = target_pages.intersection(other_pages)
        if not intersection:
            continue

        union = target_pages.union(other_pages)

        # Jaccard Score normalization
        jaccard_score = len(intersection) / len(union)

        # 5. Score unliked pages
        unliked_pages = other_pages - target_pages
        for page in unliked_pages:
            page_suggestions[page] = (
                page_suggestions.get(page, 0.0) + jaccard_score
            )

    # 6. Sort by score descending
    sorted_pages = sorted(
        page_suggestions.items(), key=lambda x: x[1], reverse=True
    )
    return sorted_pages


In [18]:
data = load_data("C:\\Users\\FDRA\\Documents\\DATA SCIENCE\\CWHDS\\massi_data.json")
userId = 80
Page_Recommadation = pags_you_might_like_pandas(userId, data)
print(Page_Recommadation)

[(108, 3.25), (110, 2.6666666666666665), (107, 2.083333333333333), (105, 1.9166666666666665), (103, 1.6666666666666665), (101, 1.5), (106, 1.5), (104, 1.0), (109, 0.75)]


# Optimized Logic (Jaccard Similarity + Pandas)

In [2]:
import json
import pandas as pd

In [3]:
def load_data(filename):
    with open(filename, 'r') as file:
        return json.load(file)

In [8]:
def pags_you_might_like_pandas(target_user_id, data):
    # 1. Flatten the JSON data into a clean DataFrame
    records = []
    for user in data['user']:
        uid = user['id']
        for page in user['liked_pages']:
            records.append({'user_is': uid, 'page_id': page_id})
            
    df = pd.DataFrame(records)
    
    # If target user doesn't exist or data is empty, exit early
    if df.empty or target_user_id not in df["user_id"].values:
        return []

    # 2. Get target user pages
    target_pages = set(df[df["user_id"] == target_user_id]["page_id"])

    # 3. Create a user-page interaction set dictionary
    user_pages_dict = df.groupby("user_id")["page_id"].apply(set).to_dict()

    page_suggestions = {}
    

    # 4. Compute Jaccard Similarity weights
    for other_user_id, other_pages in user_pages_dict.items():
        if other_user_id == target_user_id:
            continue

        intersection = target_pages.intersection(other_pages)
        if not intersection:
            continue

        union = target_pages.union(other_pages)

        # Jaccard Score normalization
        jaccard_score = len(intersection) / len(union)

        # 5. Score unliked pages
        unliked_pages = other_pages - target_pages
        for page in unliked_pages:
            page_suggestions[page] = (
                page_suggestions.get(page, 0.0) + jaccard_score
            )

    # 6. Sort by score descending
    sorted_pages = sorted(
        page_suggestions.items(), key=lambda x: x[1], reverse=True
    )
    return sorted_pages



In [12]:

data = load_data(
    "C:\\Users\\FDRA\\Documents\\DATA SCIENCE\\CWHDS\\massi_data.json"
)
userId = 80
Page_Recommendation = pags_you_might_like_pandas(userId, data)
print(Page_Recommendation)

[(108, 3.25), (110, 2.6666666666666665), (107, 2.083333333333333), (105, 1.9166666666666665), (103, 1.6666666666666665), (101, 1.5), (106, 1.5), (104, 1.0), (109, 0.75)]


In [14]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity


def load_data(filename):
    with open(filename) as f:
        return json.load(f)


def pages_you_might_like_matrix(target_user_id, data):
    # 1. Flatten the JSON data
    records = []
    for user in data["users"]:
        uid = user["id"]
        for page in user["liked_pages"]:
            records.append({"user_id": uid, "page_id": page})

    df = pd.DataFrame(records)

    # Validate target user presence
    if df.empty or target_user_id not in df["user_id"].values:
        return []

    # 2. Encode users and pages to continuous integer indices for sparse matrix
    df["user_code"] = df["user_id"].astype("category").cat.codes
    df["page_code"] = df["page_id"].astype("category").cat.codes

    # Create mapping dictionaries to reverse-lookup IDs later
    user_id_to_code = dict(zip(df["user_id"], df["user_code"]))
    page_code_to_id = dict(zip(df["page_code"], df["page_id"]))

    num_users = df["user_code"].nunique()
    num_pages = df["page_code"].nunique()

    # 3. Build the Compressed Sparse Row (CSR) Binary Matrix
    # This structure uses virtually zero memory for 0s
    user_item_matrix = csr_matrix(
        (np.ones(len(df)), (df["user_code"], df["page_code"])),
        shape=(num_users, num_pages),
        dtype=np.float32,
    )

    # 4. Compute User-to-User Cosine Similarity Matrix
    # Cosine Similarity handles implicit normalization like Jaccard
    user_sim = cosine_similarity(user_item_matrix, dense_output=False)

    # 5. Extract the targeted row for our specific user
    target_code = user_id_to_code[target_user_id]
    target_user_sim = user_sim[target_code].toarray().flatten()

    # Mask out the target user themselves so they don't recommend to themselves
    target_user_sim[target_code] = 0

    # 6. Calculate raw item recommendation scores using vector dot-product
    # Score Vector = User Similarity Vector * User-Item Matrix
    recommendation_scores = target_user_sim.dot(user_item_matrix.toarray())

    # 7. Mask out pages the target user has already liked
    target_user_profile = user_item_matrix[target_code].toarray().flatten()
    recommendation_scores[target_user_profile > 0] = 0

    # 8. Sort and structure results
    recommended_page_codes = np.argsort(recommendation_scores)[::-1]

    recommendations = []
    for code in recommended_page_codes:
        score = float(recommendation_scores[code])
        if score > 0:  # Only include pages with an active recommendation weight
            recommendations.append((page_code_to_id[code], score))

    return recommendations


# Execution
data = load_data(
    "C:\\Users\\FDRA\\Documents\\DATA SCIENCE\\CWHDS\\massi_data.json"
)
userId = 80
Page_Recommendation = pages_you_might_like_matrix(userId, data)
print(Page_Recommendation)


[(108, 5.939157009124756), (110, 4.568914413452148), (107, 4.07735013961792), (105, 3.65470027923584), (103, 3.15470027923584), (101, 3.0), (106, 2.707106828689575), (104, 1.7071068286895752), (109, 1.5)]
